In [0]:
# Widget Fallback for Secret Salt Handling
dbutils.widgets.text("hmac_secret_key", "EbayMarketplace2026SaltKey!", "HMAC Secret Salt")

def get_secret_salt():
    try:
        # Tries Databricks Secret Scope first
        return dbutils.secrets.get(scope="ebay_secrets", key="hmac_key")
    except Exception:
        # Falls back to widget parameter on Free Edition
        return dbutils.widgets.get("hmac_secret_key")

In [0]:
import datetime
import uuid
from pyspark.sql.functions import col, current_timestamp, lit, sha2, concat, expr, regexp_replace, to_timestamp, row_number
from pyspark.sql.window import Window
from pyspark.sql.types import (
    StructType, StructField, StringType, LongType, ArrayType, DecimalType, TimestampType
)

# 1. CATALOG & TABLE NAMES
CATALOG = "workspace"
SCHEMA = "ebay_pipeline"

TABLE_LOGS = f"{CATALOG}.{SCHEMA}.pipeline_execution_logs"
TABLE_BRONZE_FULL = f"{CATALOG}.{SCHEMA}.bronze_ebay_listings_full"
TABLE_BRONZE_INC = f"{CATALOG}.{SCHEMA}.bronze_ebay_listings_incremental"
TABLE_SILVER_LISTINGS = f"{CATALOG}.{SCHEMA}.silver_listings"
TABLE_SILVER_PRICE_HIST = f"{CATALOG}.{SCHEMA}.silver_price_history"
TABLE_SILVER_SELLERS = f"{CATALOG}.{SCHEMA}.silver_sellers"
TABLE_SILVER_CATEGORIES = f"{CATALOG}.{SCHEMA}.silver_categories"
TABLE_SILVER_QUARANTINE = f"{CATALOG}.{SCHEMA}.silver_quarantine"

# 2. EXPLICIT SCHEMA-ON-READ (BRONZE)
item_schema = StructType([
    StructField("itemId", StringType(), True),
    StructField("title", StringType(), True),
    StructField("categories", ArrayType(StructType([
        StructField("categoryId", StringType(), True),
        StructField("categoryName", StringType(), True)
    ])), True),
    StructField("price", StructType([
        StructField("value", StringType(), True),
        StructField("currency", StringType(), True)
    ]), True),
    StructField("condition", StringType(), True),
    StructField("buyingOptions", ArrayType(StringType()), True),
    StructField("seller", StructType([
        StructField("username", StringType(), True),
        StructField("feedbackScore", LongType(), True),
        StructField("feedbackPercentage", StringType(), True)
    ]), True),
    StructField("itemLocation", StructType([
        StructField("country", StringType(), True),
        StructField("stateOrProvince", StringType(), True),
        StructField("city", StringType(), True),
        StructField("postalCode", StringType(), True)
    ]), True),
    StructField("itemCreationDate", StringType(), True),
    StructField("itemWebUrl", StringType(), True)
])

raw_response_schema = StructType([
    StructField("total", LongType(), True),
    StructField("itemSummaries", ArrayType(item_schema), True)
])

# 3. HELPER FUNCTIONS
def get_secret_salt():
    try:
        return dbutils.secrets.get(scope="ebay_secrets", key="hmac_key")
    except:
        return "DEFAULT_HMAC_SALT_KEY_FALLBACK"